<a href="https://colab.research.google.com/github/lamia-khd/react-router/blob/main/calcule2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time # Imports the built-in module used to measure elapsed time
import numpy as np # Imports the NumPy library for fast numerical operations
from concurrent.futures import ThreadPoolExecutor # Imports the tool to manage multi-threading

def read_config(): # Defines the function that simulates reading the initial settings
    """Settings file: read first, by one worker only (2 s).""" # Documentation string for the setup function
    time.sleep(2) # Pauses the program for 2 seconds to simulate sequential read time

def download(seconds): # Defines the function simulating the file download process
    """Download one shop's sales file (it takes some seconds).""" # Documentation string for the task
    time.sleep(seconds) # Pauses for the amount of seconds passed as an argument

files = [1, 1, 1, 1, 1, 1, 1, 1] # List representing today's 8 files, each taking 1 second to download
tomorrow = [4, 1, 1, 1, 1, 1, 1, 2] # List representing tomorrow's files with varying download times
rng = np.random.default_rng(0) # Initializes the random number generator with a fixed seed
amounts = rng.integers(100, 5000, 2_000_000) # Generates an array of 2 million random sales figures

# Q1
t0 = time.perf_counter() # Starts the stopwatch for the single-worker execution
read_config() # Executes the mandatory 2-second sequential configuration read
for f in files: # Loops through every file in today's list one by one
    download(f) # Downloads the current file in the sequence
T1 = time.perf_counter() - t0 # Calculates the total execution time for 1 worker
print(f"Q1 1 worker: {T1:.1f} s") # Prints the resulting time

# Q2
t0 = time.perf_counter() # Resets and starts the stopwatch for the parallel execution
read_config() # Executes the sequential configuration step again
with ThreadPoolExecutor(max_workers=4) as team: # Creates a team of 4 independent workers
    list(team.map(download, files)) # Assigns the download tasks for the 8 files to the team
Tp = time.perf_counter() - t0 # Calculates the total execution time with 4 workers
print(f"Q2 4 workers: {Tp:.1f} s, speed-up {T1 / Tp:.1f}") # Prints the time and the calculated speed-up

# Q3
part_alone = 2.0 # Defines the time taken by the indivisible sequential portion
print(f"Q3 alone: {part_alone:.1f} s, best possible: {T1 / part_alone:.1f}") # Prints max theoretical speed-up

# Q4
t_loop = time.perf_counter() # Starts the timer for the standard Python loop addition
total_loop = sum(amounts) # Calculates the sum using Python's built-in sum function
loop_time = time.perf_counter() - t_loop # Records the time taken by the standard loop

t_numpy = time.perf_counter() # Starts the timer for the optimized NumPy addition
total_numpy = amounts.sum() # Calculates the sum using NumPy's highly optimized SIMD method
numpy_time = time.perf_counter() - t_numpy # Records the time taken by NumPy
print(f"Q4 loop {loop_time:.4f} s, numpy {numpy_time:.4f} s, same total: {total_loop == total_numpy}") # Prints comparison

# Q5
t0 = time.perf_counter() # Starts the stopwatch for tomorrow's uneven schedule
read_config() # Executes the required 2-second setup
with ThreadPoolExecutor(max_workers=4) as team: # Creates a team of 4 workers for the new workload
    list(team.map(download, tomorrow)) # Executes tomorrow's varied tasks in parallel
print(f"Q5 tomorrow, 4 workers: {time.perf_counter() - t0:.1f} s") # Prints the actual completion time

Q1 1 worker: 10.0 s
Q2 4 workers: 4.0 s, speed-up 2.5
Q3 alone: 2.0 s, best possible: 5.0
Q4 loop 0.1574 s, numpy 0.0016 s, same total: True
Q5 tomorrow, 4 workers: 6.0 s
